In [ ]:
import logging

import openeo.processes
from utils import urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

In [ ]:
# threshold applied to probability of cropland
cropland_probability_threshold = 0.1

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

# Load Uganda ADM-4 boundaries

geoboundaries URL https://www.geoboundaries.org/api/current/gbHumanitarian/UGA/ADM4/

In [ ]:
ADM_BOUNDARIES_URL = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbHumanitarian/UGA/ADM4/geoBoundaries-UGA-ADM4.geojson"

adm_boundaries = connection.load_url(
    ADM_BOUNDARIES_URL,
    format="GeoJSON",
)

In [ ]:
adm_boundaries.metadata.dimension_names()

In [ ]:
process_graph_results.append(
    adm_boundaries.save_result(
        format="GeoJSON",
        options={
            "filename_prefix": "0000_adm_boundaries",
        },
    )
)

In [ ]:
# adm_boundaries = adm_boundaries.filter_bbox(extent=spatial_extent)
# OpenEoApiError: [400] ProcessParameterInvalid: The value passed for parameter 'data' in process 'filter_bbox' is invalid: Expected raster cube but got vector cube.

In [ ]:
vectorcube_filter_bbox_udf = openeo.UDF.from_file(
    "../udf/vectorcube_filter_bbox.py",
    runtime="Python",
    version="3.11",
    # context set here is ignored! 😠
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
adm_boundaries = adm_boundaries.apply_dimension(
    process=vectorcube_filter_bbox_udf,
    dimension="geometry",
    # try passing context here as well 🤷‍♂️
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
process_graph_results.append(
    adm_boundaries.save_result(
        format="GeoJSON",
        options={
            "filename_prefix": "0001_adm_boundaries",
        },
    )
)

# Load decimal year of deforestation

forest baseline is also packaged alongside year of deforestation

Bands: [year_of_deforestation, forest_baseline]

In [ ]:
# load results from previous batch job
JOB_ID = "j-26091510470147c4b69c962dacd24c7a"

deforestation_year = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
process_graph_results.append(
    deforestation_year.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0010_deforestation_year",
        },
    )
)

In [ ]:
deforestation_year = deforestation_year.drop_dimension("t")

In [ ]:
process_graph_results.append(
    deforestation_year.save_result(
        format="NetCDF",
        options={
            "filename_prefix": "0011_deforestation_year",
        },
    )
)

# load cropland probability

In [ ]:
cropland_probability = connection.load_stac(
    url=urls.MEAN_CROPS_STAC,
    spatial_extent=spatial_extent,
    bands=["crops"],
)

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_cropland_probability",
        },
    )
)

In [ ]:
cropland_probability = utils.drop_hidden_dimension(cropland_probability, "t")

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0031_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0031_cropland_probability",
        },
    )
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
cropland_probability = cropland_probability.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
process_graph_results.append(
    cropland_probability.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0032_cropland_probability",
        },
    )
)
process_graph_results.append(
    cropland_probability.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0032_cropland_probability",
        },
    )
)

In [ ]:
cropland_mask = cropland_probability.band("crops") > cropland_probability_threshold

In [ ]:
process_graph_results.append(
    cropland_mask.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0040_cropland_mask",
        },
    )
)
process_graph_results.append(
    cropland_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_cropland_mask",
        },
    )
)

merge cropland mask into deforestation year datacube

Bands: [year_of_deforestation, forest_baseline, cropland]

In [ ]:
cropland_mask = cropland_mask.add_dimension("bands", label="cropland", type="bands")

In [ ]:
intermediate_datacube = deforestation_year.merge_cubes(cropland_mask)

In [ ]:
process_graph_results.append(
    intermediate_datacube.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0050_intermediate_datacube",
        },
    )
)
process_graph_results.append(
    intermediate_datacube.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_intermediate_datacube",
        },
    )
)

# Generate raster KIPs

The CDSE openEO backend has basically no support for processing vector data.
Ideally I would generate each KPI, then merge them together into a single output.
However, this is not possible.

https://forum.dataspace.copernicus.eu/t/merge-vector-cubes/5425/

Essentially, `aggregate_spatial` has to be the last process,
because after that the data is a vector cube, and there is no facility to process it any futher. 😡

Here we use a UDF to generate raster KPI layers, 
where each pixel has value = pixel area (units: ha).
Ready for passing to `aggregate_spatial`.

Doing this in a UDF is an optimisation, rather than using many native openEO band math and `merge_cubes` operations that would make the process graph more complex.

In [ ]:
raster_kpis_udf = openeo.UDF.from_file(
    "../udf/raster_kpis.py",
    runtime="Python",
    version="3.11",
    context={
        "years": [2020, 2021, 2022, 2023, 2024],
        "spatial_resolution": resample_spatial_resolution,
    },
)

In [ ]:
kpis_raster = intermediate_datacube.apply_dimension(
    process=raster_kpis_udf,
    dimension="bands",
)

In [ ]:
process_graph_results.append(
    kpis_raster.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_kpis_raster",
        },
    )
)
process_graph_results.append(
    kpis_raster.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_kpis_raster",
        },
    )
)

raster stats

In [ ]:
kpis_vector = kpis_raster.aggregate_spatial(
    geometries=adm_boundaries, reducer=openeo.processes.sum
)

In [ ]:
# json format is pretty useless - no metadata (column names)
# process_graph_results.append(
#     kpis_vector.save_result(
#         format="JSON",
#     )
# )

# CSV format is also pretty useless - only indexed by feature_id
# no geometry or feature properties
# process_graph_results.append(
#     kpis_vector.save_result(
#         format="CSV",
#         options={
#             "filename_prefix": "0090_kpis_vector",
#             # https://forum.dataspace.copernicus.eu/t/access-to-geojson-properties/5418/
#             "feature_id_property": "shapeName",
#         },
#     )
# )

process_graph_results.append(
    kpis_vector.save_result(
        format="Parquet",
        options={
            "filename_prefix": "0090_kpis_vector",
        },
    )
)

# Run batch job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)

In [ ]:
job = multi_result.create_job(
    # https://forum.dataspace.copernicus.eu/t/multiresult-with-geojson/5416
    # Adds a UUID to each output, but multiple outputs overwrite each other 😠
    # job_options={"stac-version": "1.1"}
)
job.start_and_wait()
# Inspect job.logs() if it fails

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)